# FlashAttention Memory Benchmarking

Measure peak GPU memory usage for:
- Naive attention (materializes full N×N matrix)
- FlashAttention (streaming, minimal intermediate storage)

Shows the sequence length at which naive attention runs out of memory.

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path().resolve().parent / "src"))
from flash_attention import naive_attention, flash_attention_forward, FlashAttentionTriton

# Disable benchmarking to reduce variance
torch.backends.cuda.benchmark = False

In [ ]:
def measure_peak_memory(fn, num_runs=3, *args, **kwargs):
    """
    Measure peak memory for a function call with multiple runs.
    Uses torch.cuda.synchronize() to ensure accurate measurement.
    
    Args:
        fn: Function to measure
        num_runs: Number of independent measurement runs
        *args, **kwargs: Arguments to pass to fn
    
    Returns:
        Tuple of (peak_memory_gb, status_msg) or (None, error_msg)
    """
    peak_mems = []
    
    for run in range(num_runs):
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
        
        try:
            result = fn(*args, **kwargs)
            torch.cuda.synchronize()
            peak_bytes = torch.cuda.max_memory_allocated()
            peak_mems.append(peak_bytes / (1024**3))
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                return None, "OOM"
            else:
                return None, str(e)
    
    # Return max peak memory across runs (most conservative estimate)
    return max(peak_mems), None

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Total memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
print("(Memory measurements use 3 independent runs to account for variance)")

In [ ]:
# Sequence lengths to test
seq_lens = [128, 256, 512, 1024, 2048, 4096, 8192, 16384]

# Fixed params
batch = 1
heads = 8
head_dim = 64

results = {
    "seq_len": [],
    "naive_mem_gb": [],
    "flash_mem_gb": [],
    "naive_status": [],
    "flash_status": [],
}

print(f"Shape: (batch={batch}, heads={heads}, seq_len=?, head_dim={head_dim})")
print()
print(f"{'seq_len':>6} | {'naive (GB)':>12} | {'flash (GB)':>12} | {'status':>20}")
print("-" * 65)

for seq_len in seq_lens:
    Q = torch.randn(batch, heads, seq_len, head_dim, device='cuda', dtype=torch.float32)
    K = torch.randn(batch, heads, seq_len, head_dim, device='cuda', dtype=torch.float32)
    V = torch.randn(batch, heads, seq_len, head_dim, device='cuda', dtype=torch.float32)
    
    # Measure naive attention
    naive_mem, naive_err = measure_peak_memory(lambda: naive_attention(Q, K, V, is_causal=True))
    naive_status = "OK" if naive_mem is not None else naive_err
    
    # Measure flash attention (returns tuple of (output, L))
    flash_mem, flash_err = measure_peak_memory(lambda: flash_attention_forward(Q, K, V, causal=True)[0])
    flash_status = "OK" if flash_mem is not None else flash_err
    
    results["seq_len"].append(seq_len)
    results["naive_mem_gb"].append(naive_mem)
    results["flash_mem_gb"].append(flash_mem)
    results["naive_status"].append(naive_status)
    results["flash_status"].append(flash_status)
    
    naive_str = f"{naive_mem:.3f}" if naive_mem is not None else "OOM"
    flash_str = f"{flash_mem:.3f}" if flash_mem is not None else "OOM"
    status = "✓" if naive_status == "OK" and flash_status == "OK" else "✗"
    
    print(f"{seq_len:>6} | {naive_str:>12} | {flash_str:>12} | {status:>20}")

In [ ]:
# Plot memory usage
fig, ax = plt.subplots(figsize=(10, 6))

# Only plot successful runs
seq_lens_success = [s for s, ns in zip(results["seq_len"], results["naive_status"]) if ns == "OK"]
naive_mems = [m for m, ns in zip(results["naive_mem_gb"], results["naive_status"]) if ns == "OK"]

seq_lens_flash = [s for s, fs in zip(results["seq_len"], results["flash_status"]) if fs == "OK"]
flash_mems = [m for m, fs in zip(results["flash_mem_gb"], results["flash_status"]) if fs == "OK"]

ax.plot(seq_lens_success, naive_mems, 'o-', linewidth=2, markersize=8, label='Naive Attention', color='#e74c3c')
ax.plot(seq_lens_flash, flash_mems, 's-', linewidth=2, markersize=8, label='FlashAttention', color='#2ecc71')

# Mark OOM points
if len(seq_lens_success) < len(results["seq_len"]):
    oom_idx = len(seq_lens_success)
    ax.axvline(results["seq_len"][oom_idx], color='#e74c3c', linestyle='--', alpha=0.5, linewidth=2, label='Naive OOM')

ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('Peak Memory (GB)', fontsize=12)
ax.set_title('Peak GPU Memory Usage by Sequence Length', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("\n" + "="*60)
print("MEMORY BENCHMARK SUMMARY")
print("="*60)

if results["naive_status"].count("OK") > 0:
    max_naive_seq = results["seq_len"][results["naive_status"].index("OK") + results["naive_status"][results["naive_status"].index("OK"):].index("OOM")]
    print(f"\nNaive attention:")
    print(f"  Max working sequence length: {max_naive_seq}")
    max_naive_idx = [i for i, s in enumerate(results["naive_status"]) if s == "OK"][-1]
    print(f"  Peak memory at seq_len={results['seq_len'][max_naive_idx]}: {results['naive_mem_gb'][max_naive_idx]:.3f} GB")

max_flash_idx = [i for i, s in enumerate(results["flash_status"]) if s == "OK"][-1]
print(f"\nFlashAttention:")
print(f"  Max tested sequence length: {results['seq_len'][max_flash_idx]}")
print(f"  Peak memory at seq_len={results['seq_len'][max_flash_idx]}: {results['flash_mem_gb'][max_flash_idx]:.3f} GB")

if results["naive_status"].count("OK") > 0 and results["flash_status"].count("OK") > 0:
    # Find crossover point
    ok_indices = [i for i, (ns, fs) in enumerate(zip(results["naive_status"], results["flash_status"])) if ns == "OK" and fs == "OK"]
    if len(ok_indices) > 1:
        speedup = results["naive_mem_gb"][ok_indices[-1]] / results["flash_mem_gb"][ok_indices[-1]]
        print(f"\nMemory reduction at max working seq_len:")
        print(f"  {speedup:.1f}x reduction (naive vs flash)")